# mydata 数据检查

这个 notebook 用来检查 `experiment-log.md` 和 `mydata` 目录中的 CSV 文件：实验时间线、CSV 表头、列名一致性、时间字段、服务字段、请求/延迟/错误字段，以及故障文件是否已有 label。

## 从 experiment-log.md 整理出的实验信息

| 文件 | 实验 | 采集开始 | 采集结束 | 故障注入 | 故障停止 | 故障服务 | 故障类型 | 故障持续 |
|---|---|---|---|---|---|---|---|---|
| normal_train.csv | 正常训练集 | 2026-06-05 12:36:43 | 2026-06-05 13:06:43 | 无 | 无 | 无 | normal | 无 |
| normal_valid.csv | 正常验证集 | 2026-06-05 12:56:43 | 2026-06-05 13:06:43 | 无 | 无 | 无 | normal | 无 |
| fault_cpu_frontend.csv | CPU 故障 | 2026-06-05 13:16:00 | 2026-06-05 13:30:00 | 2026-06-05 13:16:00 | 2026-06-05 13:23:00 | frontend | CPU stress | 7 分钟 |
| fault_delay_cart.csv | 网络延迟 | 2026-06-05 13:33:00 | 2026-06-05 13:47:00 | 2026-06-05 13:33:00 | 2026-06-05 13:39:00 | cartservice | network delay | 6 分钟 |
| fault_kill_product.csv | Pod 杀死 | 2026-06-05 13:49:00 | 2026-06-05 14:03:00 | 2026-06-05 13:49:00 | 2026-06-05 13:56:00 | productcatalogservice | pod kill | 7 分钟 |

注意：`experiment-log.md` 中导出路径写的是 `data/...csv`，当前实际文件位于 `mydata/...csv`。

In [ ]:
from pathlib import Path
import pandas as pd

# 兼容两种打开方式：从项目根目录打开，或直接从 mydata 目录打开。
DATA_DIR = Path('.')
if not (DATA_DIR / 'experiment-log.md').exists():
    DATA_DIR = Path('mydata')

csv_files = sorted(DATA_DIR.glob('*.csv'))
print('DATA_DIR =', DATA_DIR.resolve())
print('CSV files:')
for path in csv_files:
    print('-', path.name)


In [ ]:
# 打开 experiment-log.md，显式使用 UTF-8，避免中文乱码。
log_text = (DATA_DIR / 'experiment-log.md').read_text(encoding='utf-8')
print(log_text)


In [ ]:
# 查看每个 CSV 的第一行表头。
headers = {}
for path in csv_files:
    first_line = path.read_text(encoding='utf-8').splitlines()[0]
    headers[path.name] = first_line.split(',')
    print(f'--- {path.name}')
    print(first_line)


In [ ]:
# 把表头变成表格，方便横向比较。
header_summary = pd.DataFrame([
    {'file': name, 'n_columns': len(cols), 'columns': ', '.join(cols)}
    for name, cols in headers.items()
])
display(header_summary)


In [ ]:
# 1. 检查每个 CSV 的列名是否一样。
reference_name = csv_files[0].name
reference_cols = headers[reference_name]

column_compare = []
for name, cols in headers.items():
    missing = [c for c in reference_cols if c not in cols]
    extra = [c for c in cols if c not in reference_cols]
    same_order = cols == reference_cols
    column_compare.append({
        'file': name,
        'same_columns_and_order_as_reference': same_order,
        'missing_vs_reference': missing,
        'extra_vs_reference': extra,
    })

column_compare_df = pd.DataFrame(column_compare)
display(column_compare_df)
print('Reference file:', reference_name)
print('All CSV columns identical and in the same order:', column_compare_df['same_columns_and_order_as_reference'].all())


In [ ]:
# 2-5. 检查 timestamp/time、service、requests/latency/errors、label 字段。
def has_any(cols, keywords):
    return any(any(k in c.lower() for k in keywords) for c in cols)

checks = []
for name, cols in headers.items():
    lower_cols = [c.lower() for c in cols]
    checks.append({
        'file': name,
        'has_timestamp': 'timestamp' in lower_cols,
        'has_time': 'time' in lower_cols,
        'has_service_column': 'service' in lower_cols,
        'has_requests_field': has_any(cols, ['request', 'requests', 'req']),
        'has_latency_field': has_any(cols, ['latency', 'duration', 'delay']),
        'has_errors_field': has_any(cols, ['error', 'errors', 'failure', 'fail']),
        'has_label_field': has_any(cols, ['label', 'class', 'target', 'fault_type', 'is_fault']),
    })

checks_df = pd.DataFrame(checks)
display(checks_df)


In [ ]:
# 当前数据没有独立 service 列，但服务名写在指标列名前缀里，比如 frontend_cpu、cartservice_mem。
metric_suffixes = ('_cpu', '_mem', '_net_rx', '_net_tx')
services = sorted({
    col.rsplit('_', 1)[0]
    for col in reference_cols
    if col != 'timestamp' and col.endswith(metric_suffixes)
})
print('Inferred services from metric column prefixes:')
for svc in services:
    print('-', svc)


## 当前结论

1. 每个 CSV 的列名一样，而且顺序一致。
2. 每个 CSV 都有 `timestamp` 字段。
3. 没有单独的 `service` 字段；服务名体现在列名前缀中，例如 `frontend_cpu`、`cartservice_net_rx`。
4. 没有 `requests`、`latency`、`errors` 这类业务/调用指标字段；目前是 CPU、内存、网络收发指标。
5. 故障文件里没有 `label`、`normal`、`fault` 这类标签列。后续训练/评估如果需要标签，需要根据 `experiment-log.md` 的时间段和文件名另外生成。